Importation des bibliothèques

In [1]:
import os
import random
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import copy #pour copier base_model
import numpy as np
import pandas as pd 
import config

In [2]:
from models import SimpleCNN, CNN_MCdropout
from dataset import load_cifar10
from train import train_model, evaluate
from utils import mc_predict_mean_probs, generate_mc_outputs
from accuracy import accuracy_threshold, monotonic_rearrangement, isotonic_regression, monotonicity_penalty

In [3]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [4]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def set_seed(seed):
    import random
    import torch
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [5]:
trainloader, valloader, testloader, classes = load_cifar10(batch_size=128, val_ratio=0.1)

Test manuel de plusieurs dico_layers et stockage des résultats

In [6]:
# Liste des configurations à tester (remplace la liste par 3 variables)
dico_layers_1 = {"conv1": 0.1, "conv2": 0.1, "conv3": 0.1, "fc1": 0.2}
dico_layers_2 = {"conv1": 0.5, "fc1": 0.5}
dico_layers_3 = {"fc1": 0.8}

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

base_model_mc1 = copy.deepcopy(base_model)
base_model_mc2 = copy.deepcopy(base_model)
base_model_mc3 = copy.deepcopy(base_model)

model_1 = CNN_MCdropout(base_model_mc1, dico_layers=dico_layers_1, before=False).to(device)
model_2 = CNN_MCdropout(base_model_mc2, dico_layers=dico_layers_2, before=True).to(device)
model_3 = CNN_MCdropout(base_model_mc3, dico_layers=dico_layers_3, before=False).to(device)

# Évaluation finale du modèle sans dropout
test_loss, test_acc = evaluate(model_1, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267


MC Dropout prédiction

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [8]:
results = []

X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)

probs1, t1 = mc_predict_mean_probs(model_1, X, T=1000, verbose=True)
Y_hat1 = probs1.argmax(1)
acc1 = (Y_hat1 == Y).float().mean().item()

probs2, t2 = mc_predict_mean_probs(model_2, X, T=1000, verbose=True)
Y_hat2 = probs2.argmax(1)
acc2 = (Y_hat2 == Y).float().mean().item()

probs3, t3 = mc_predict_mean_probs(model_3, X, T=1000, verbose=True)
Y_hat3 = probs3.argmax(1)
acc3 = (Y_hat3 == Y).float().mean().item()

results.append({
    "model": "model_1",
    "Y_hat": Y_hat1.cpu().numpy(),
    "Y": Y.cpu().numpy(),
    "acc": acc1
})
results.append({
    "model": "model_2",
    "Y_hat": Y_hat2.cpu().numpy(),
    "Y": Y.cpu().numpy(),
    "acc": acc2
})
results.append({
    "model": "model_3",
    "Y_hat": Y_hat3.cpu().numpy(),
    "Y": Y.cpu().numpy(),
    "acc": acc3
})


100%|██████████| 1000/1000 [00:13<00:00, 73.11it/s]


Temps total: 13.68 s  |  Temps moyen par passe: 0.0137 s


100%|██████████| 1000/1000 [00:13<00:00, 72.81it/s]


Temps total: 13.75 s  |  Temps moyen par passe: 0.0138 s


100%|██████████| 1000/1000 [00:11<00:00, 83.42it/s]

Temps total: 12.00 s  |  Temps moyen par passe: 0.0120 s


Métriques

In [9]:
results_metrics = []

models = [
    ("model_1", model_1),
    ("model_2", model_2),
    ("model_3", model_3)
]

X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)

user_metrics = input(
    "Quelles métriques voulez-vous calculer ?\n"
    "Options disponibles : mc_estimate, variance_predicted, variance_max, predictive_entropy_predicted, predictive_entropy_max, relative_norm\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]

print_metrics = False  # <-- Ajoutez ce booléen ici

for name, model in models:
    outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
        model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False
    )
    metrics_entry = {
        "model": name,
        "metric_values": metric_values
    }
    results_metrics.append(metrics_entry)
    
    if print_metrics:
        print(f"\nModèle : {name}")
        print(f"Temps total des forward passes : {elapsed_forward:.2f} s\n")
        for metric in user_metrics:
            metric_result = metric_values[metric]
            for r in results:
                if r["model"] == name:
                    if "metrics" not in r:
                        r["metrics"] = {}
                    r["metrics"][metric] = metric_result
                    break

            print(f"--- Métrique choisie : {metric} ---")
            print(f"Temps de calcul : {elapsed_metrics[metric]:.6f} s")
            if metric == "mc_estimate":
                print(f"Distribution de probabilités moyennes (par échantillon) :\n{metric_result}\n")
            elif metric == "variance_max":
                print(f"Variance moyenne (logits) : {metric_values['variance_max_mean']:.6f}")
                print(f"Variance par échantillon :\n{metric_result}\n")
            elif metric == "variance_predicted":
                print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
                print(f"Variance softmax par échantillon :\n{metric_result}\n")
            elif metric == "predictive_entropy_max":
                print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_max_mean']:.6f}")
                print(f"Entropie prédictive par échantillon :\n{metric_result}\n")
            elif metric == "predictive_entropy_predicted":
                print(f"Entropie attendue moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
                print(f"Entropie attendue par échantillon :\n{metric_result}\n")
            elif metric == "relative_norm":
                print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
                print(f"Norme relative par échantillon :\n{metric_result}\n")
            else:
                print(f"Résultat brut :\n{metric_result}\n")
    else:
        # Toujours mettre à jour results même si on n'affiche pas
        for metric in user_metrics:
            metric_result = metric_values[metric]
            for r in results:
                if r["model"] == name:
                    if "metrics" not in r:
                        r["metrics"] = {}
                    r["metrics"][metric] = metric_result
                    break

Fonctions d'accuracy

Seuil de variance_predicted

In [10]:
accuracy_results_vp = []

for res in results:
    thresholds_vp, accuracies_vp = accuracy_threshold(
        res["Y_hat"], res["Y"], res["metrics"]["variance_predicted"],
        metric_name="variance predicted", color="royalblue", display=False
    )
    accuracy_results_vp.append({
        "model": res["model"],
        "thresholds_vp": thresholds_vp,
        "accuracies_vp": accuracies_vp
    })

In [11]:
iso_accuracies_results_vp = []

for acc_res in accuracy_results_vp:
    iso_accuracies_vp = isotonic_regression(
        acc_res["thresholds_vp"], acc_res["accuracies_vp"], color='royalblue', display=False
    )
    iso_accuracies_results_vp.append({
        "model": acc_res["model"],
        "iso_accuracies_vp": iso_accuracies_vp
    })

In [12]:
rearranged_accuracies_results_vp = []

for acc_res in accuracy_results_vp:
    rearranged_accuracies_vp = monotonic_rearrangement(
        acc_res["accuracies_vp"], acc_res["thresholds_vp"], acc_res["accuracies_vp"], color='royalblue', display=False
    )
    rearranged_accuracies_results_vp.append({
        "model": acc_res["model"],
        "rearranged_accuracies_vp": rearranged_accuracies_vp
    })

In [13]:
penalties_vp = []

for acc_res, iso_res, rearr_res in zip(accuracy_results_vp, iso_accuracies_results_vp, rearranged_accuracies_results_vp):
    penalty_iso = monotonicity_penalty(acc_res["thresholds_vp"], acc_res["accuracies_vp"], iso_res["iso_accuracies_vp"])
    penalty_rearr = monotonicity_penalty(acc_res["thresholds_vp"], acc_res["accuracies_vp"], rearr_res["rearranged_accuracies_vp"])
    penalties_vp.append({
        "model": acc_res["model"],
        "penalty_isotone": penalty_iso,
        "penalty_rearrangement": penalty_rearr
    })
    print(f"{acc_res['model']} - Pénalité (isotone): {penalty_iso:.6f}, (réarrangement): {penalty_rearr:.6f}")

model_1 - Pénalité (isotone): 0.001445, (réarrangement): 0.004991
model_2 - Pénalité (isotone): 0.007452, (réarrangement): 0.081084
model_3 - Pénalité (isotone): 0.003268, (réarrangement): 0.006377


Seuil de variance_max

In [14]:
accuracy_results_vm = []

for res in results:
    thresholds_vm, accuracies_vm = accuracy_threshold(
        res["Y_hat"], res["Y"], res["metrics"]["variance_max"],
        metric_name="variance max", color="seagreen", display=False
    )
    accuracy_results_vm.append({
        "model": res["model"],
        "thresholds_vm": thresholds_vm,
        "accuracies_vm": accuracies_vm
    })

In [15]:
iso_accuracies_results_vm = []

for acc_res in accuracy_results_vm:
    iso_accuracies_vm = isotonic_regression(
        acc_res["thresholds_vm"], acc_res["accuracies_vm"], color='seagreen', display=False
    )
    iso_accuracies_results_vm.append({
        "model": acc_res["model"],
        "iso_accuracies_vm": iso_accuracies_vm
    })

In [16]:
rearranged_accuracies_results_vm = []

for acc_res in accuracy_results_vm:
    rearranged_accuracies_vm = monotonic_rearrangement(
        acc_res["accuracies_vm"], acc_res["thresholds_vm"], acc_res["accuracies_vm"], color='seagreen', display=False
    )
    rearranged_accuracies_results_vm.append({
        "model": acc_res["model"],
        "rearranged_accuracies_vm": rearranged_accuracies_vm
    })

In [17]:
penalties_vm = []

for acc_res, iso_res, rearr_res in zip(accuracy_results_vm, iso_accuracies_results_vm, rearranged_accuracies_results_vm):
    penalty_iso = monotonicity_penalty(acc_res["thresholds_vm"], acc_res["accuracies_vm"], iso_res["iso_accuracies_vm"])
    penalty_rearr = monotonicity_penalty(acc_res["thresholds_vm"], acc_res["accuracies_vm"], rearr_res["rearranged_accuracies_vm"])
    penalties_vm.append({
        "model": acc_res["model"],
        "penalty_isotone": penalty_iso,
        "penalty_rearrangement": penalty_rearr
    })
    print(f"{acc_res['model']} - Pénalité (isotone): {penalty_iso:.6f}, (réarrangement): {penalty_rearr:.6f}")

model_1 - Pénalité (isotone): 0.000062, (réarrangement): 0.000131
model_2 - Pénalité (isotone): 0.000167, (réarrangement): 0.000512
model_3 - Pénalité (isotone): 0.000272, (réarrangement): 0.000791


Seuil de PE_predicted

In [18]:
accuracy_results_pep = []

for res in results:
    thresholds_pep, accuracies_pep = accuracy_threshold(
        res["Y_hat"], res["Y"], res["metrics"]["predictive_entropy_predicted"],
        metric_name="predictive entropy predicted", color="deeppink", display=False
    )
    accuracy_results_pep.append({
        "model": res["model"],
        "thresholds_pep": thresholds_pep,
        "accuracies_pep": accuracies_pep
    })

In [19]:
iso_accuracies_results_pep = []

for acc_res in accuracy_results_pep:
    iso_accuracies_pep = isotonic_regression(
        acc_res["thresholds_pep"], acc_res["accuracies_pep"], color='deeppink', display=False
    )
    iso_accuracies_results_pep.append({
        "model": acc_res["model"],
        "iso_accuracies_pep": iso_accuracies_pep
    })

In [20]:
rearranged_accuracies_results_pep = []

for acc_res in accuracy_results_pep:
    rearranged_accuracies_pep = monotonic_rearrangement(
        acc_res["accuracies_pep"], acc_res["thresholds_pep"], acc_res["accuracies_pep"], color='deeppink', display=False
    )
    rearranged_accuracies_results_pep.append({
        "model": acc_res["model"],
        "rearranged_accuracies_pep": rearranged_accuracies_pep
    })

In [21]:
penalties_pep = []

for acc_res, iso_res, rearr_res in zip(accuracy_results_pep, iso_accuracies_results_pep, rearranged_accuracies_results_pep):
    penalty_iso = monotonicity_penalty(acc_res["thresholds_pep"], acc_res["accuracies_pep"], iso_res["iso_accuracies_pep"])
    penalty_rearr = monotonicity_penalty(acc_res["thresholds_pep"], acc_res["accuracies_pep"], rearr_res["rearranged_accuracies_pep"])
    penalties_pep.append({
        "model": acc_res["model"],
        "penalty_isotone": penalty_iso,
        "penalty_rearrangement": penalty_rearr
    })
    print(f"{acc_res['model']} - Pénalité (isotone): {penalty_iso:.6f}, (réarrangement): {penalty_rearr:.6f}")

model_1 - Pénalité (isotone): 0.002983, (réarrangement): 0.011498
model_2 - Pénalité (isotone): 0.022669, (réarrangement): 0.175747
model_3 - Pénalité (isotone): 0.004424, (réarrangement): 0.008877


Seuil de PE_max

In [22]:
accuracy_results_pem = []

for res in results:
    thresholds_pem, accuracies_pem = accuracy_threshold(
        res["Y_hat"], res["Y"], res["metrics"]["predictive_entropy_max"],
        metric_name="predictive entropy max", color="darkorchid", display=False
    )
    accuracy_results_pem.append({
        "model": res["model"],
        "thresholds_pem": thresholds_pem,
        "accuracies_pem": accuracies_pem
    })

In [23]:
iso_accuracies_results_pem = []

for acc_res in accuracy_results_pem:
    iso_accuracies_pem = isotonic_regression(
        acc_res["thresholds_pem"], acc_res["accuracies_pem"], color='darkorchid', display=False
    )
    iso_accuracies_results_pem.append({
        "model": acc_res["model"],
        "iso_accuracies_pem": iso_accuracies_pem
    })

In [24]:
rearranged_accuracies_results_pem = []

for acc_res in accuracy_results_pem:
    rearranged_accuracies_pem = monotonic_rearrangement(
        acc_res["accuracies_pem"], acc_res["thresholds_pem"], acc_res["accuracies_pem"], color='darkorchid', display=False
    )
    rearranged_accuracies_results_pem.append({
        "model": acc_res["model"],
        "rearranged_accuracies_pem": rearranged_accuracies_pem
    })

In [25]:
penalties_pem = []

for acc_res, iso_res, rearr_res in zip(accuracy_results_pem, iso_accuracies_results_pem, rearranged_accuracies_results_pem):
    penalty_iso = monotonicity_penalty(acc_res["thresholds_pem"], acc_res["accuracies_pem"], iso_res["iso_accuracies_pem"])
    penalty_rearr = monotonicity_penalty(acc_res["thresholds_pem"], acc_res["accuracies_pem"], rearr_res["rearranged_accuracies_pem"])
    penalties_pem.append({
        "model": acc_res["model"],
        "penalty_isotone": penalty_iso,
        "penalty_rearrangement": penalty_rearr
    })
    print(f"{acc_res['model']} - Pénalité (isotone): {penalty_iso:.6f}, (réarrangement): {penalty_rearr:.6f}")

model_1 - Pénalité (isotone): 0.000721, (réarrangement): 0.001841
model_2 - Pénalité (isotone): 0.006483, (réarrangement): 0.019105
model_3 - Pénalité (isotone): 0.039718, (réarrangement): 0.095270


Grid search sur dico_layers et stockage des résultats

In [26]:
# Dictionnaires pour chaque type de pénalité
penalties_dict = {
    "variance_predicted": penalties_vp,
    "variance_max": penalties_vm,
    "predictive_entropy_predicted": penalties_pep,
    "predictive_entropy_max": penalties_pem,
}

# 2. Récupérer les métriques moyennes pour chaque modèle
rows = []
for r in results:
    model = r["model"]
    if "metrics" in r:
        for metric, value in r["metrics"].items():
            # On cherche le type de métrique pour trouver la bonne pénalité
            if metric.startswith("variance_predicted"):
                pen_list = penalties_vp
            elif metric.startswith("variance_max"):
                pen_list = penalties_vm
            elif metric.startswith("predictive_entropy_predicted"):
                pen_list = penalties_pep
            elif metric.startswith("predictive_entropy_max"):
                pen_list = penalties_pem
            else:
                pen_list = []

            # On récupère la pénalité pour ce modèle et cette métrique
            penalty_iso = None
            penalty_rearr = None
            for p in pen_list:
                if p["model"] == model:
                    penalty_iso = p["penalty_isotone"]
                    penalty_rearr = p["penalty_rearrangement"]
                    break

            # Si c'est un tensor, on prend la moyenne
            if isinstance(value, torch.Tensor):
                mean_value = value.mean().item()
                rows.append([
                    model,
                    metric + "_mean",
                    penalty_iso,
                    penalty_rearr,
                    mean_value
                ])
            # Si c'est un float/int
            elif isinstance(value, (float, int, np.floating, np.integer)):
                rows.append([
                    model,
                    metric,
                    penalty_iso,
                    penalty_rearr,
                    value
                ])
            # Si c'est un dict avec une clé 'mean'
            elif isinstance(value, dict) and "mean" in value:
                rows.append([
                    model,
                    metric + "_mean",
                    penalty_iso,
                    penalty_rearr,
                    value["mean"]
                ])

# 3. Créer le DataFrame multi-indexé
df = pd.DataFrame(rows, columns=["model", "metric", "penalty_isotone", "penalty_rearrangement", "value"])
if not df.empty:
    df = df.set_index(["model", "metric"])[["value", "penalty_isotone", "penalty_rearrangement"]]
    
    def highlight_min(s):
        is_min = s == s.min()
        return ['background-color: #228B22; font-weight: bold' if v else '' for v in is_min]
    
    # On applique le style par colonne, pour chaque métrique (donc groupby niveau 1)
    styled = df.groupby(level=1).apply(lambda g: g.style.apply(highlight_min))
    display(df.style.apply(lambda x: highlight_min(x), subset=["value"], axis=0)
                .apply(lambda x: highlight_min(x), subset=["penalty_isotone"], axis=0)
                .apply(lambda x: highlight_min(x), subset=["penalty_rearrangement"], axis=0))
else:
    print("Aucune donnée à afficher (vérifie la structure de results et metrics).")